# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第15章：測定の信頼性と情報を持つ欠測


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Landis & Koch (1977) kappaの目安／Shrout & Fleiss (1979) ICCの型／音声研究における測定可用性バイアス。

In [ ]:
install.packages(c("irr", "psych", "lme4"), repos = "https://cloud.r-project.org")
library(irr)     # kappa係数
library(psych)   # 級内相関係数 ICC
library(lme4)
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ 分析の前に、測定そのものを疑う：

ここまでの章は、データが正しく測れていることを暗黙の前提にしてきた。しかし音声研究のデータは、人手のアノテーションか自動抽出アルゴリズムを通って生まれる。**どちらも失敗する**。境界の付け方は人によって違い、F0の自動抽出は声質によって失敗する。

この章では2つの問題を扱う。第一に、**複数の人が同じものを測ったとき、どれくらい一致しているか**を数値化する方法。第二に、**測定に失敗したデータが「たまたま」失敗したのではないとき**に何が起きるかである。後者は音声研究に固有と言ってよいほど重要で、しかも見落とされやすい。

■ 1. 評定者間の一致度：一致率をそのまま報告してはいけない：

In [ ]:
set.seed(15)
n_item <- 60

# 「本当の値」を決めて、そこに評定者ごとの癖と誤差を乗せる
truth <- rnorm(n_item, 0, 1)
make_rating <- function(bias, noise) {
  as.integer(cut(truth + bias + rnorm(n_item, 0, noise),
                 breaks = c(-Inf, -1, 0, 1, Inf), labels = 1:4))
}
ratings <- data.frame(
  r1 = make_rating(0,   0.4),
  r2 = make_rating(0,   0.4),
  r3 = make_rating(0.6, 0.4)   # r3 は全体的に甘めに付ける評定者
)

cat("単純一致率 r1-r2      :", round(mean(ratings$r1 == ratings$r2), 3), "\n")
cat("Cohen's kappa r1-r2   :", round(kappa2(ratings[, c("r1","r2")])$value, 3), "\n")
cat("Cohen's kappa r1-r3   :", round(kappa2(ratings[, c("r1","r3")])$value, 3), "\n")
cat("重み付きkappa r1-r3   :",
    round(kappa2(ratings[, c("r1","r3")], weight = "squared")$value, 3), "\n")

cat("\n級内相関係数 ICC:\n")
# ★ICC2（単一評定者、two-way random、絶対一致）とICC2k（3人平均、
#   同じtwo-way randomモデルのk人平均版）を並べる。psych::ICC()の結果は
#   [1]=ICC1, [2]=ICC2, [3]=ICC3, [4]=ICC1k, [5]=ICC2k, [6]=ICC3k の順で
#   返るので、「単一評定者→その平均」を比べるにはモデルの型を揃えて
#   [2]と[5]を選ぶ。[2]と[4]（ICC1k）を並べると、two-way randomモデルの
#   単一評定者値と、別モデルであるone-way randomの平均値を比べることに
#   なってしまう——見かけ上は近い値が出ることがあるが、モデルの前提
#   （評定者を無作為抽出とみなすかどうかの一方向/二方向の違い）が違うため、
#   同じ「平均化の効果」を測ったことにならない。
print(ICC(ratings)$results[c(2, 5), c("type", "ICC", "lower bound", "upper bound")])

# --- 実行結果 ---
# 単純一致率 r1-r2      : 0.55
# Cohen's kappa r1-r2   : 0.391
# Cohen's kappa r1-r3   : 0.211
# 重み付きkappa r1-r3   : 0.622
#
# 級内相関係数 ICC:
#                        type       ICC lower bound upper bound
# Single_random_raters   ICC2 0.6592413   0.4566045   0.7912765
# Average_random_raters ICC2k 0.8530251   0.7159770   0.9191794
#
# 【出力の読み方】
# ・**単純一致率55%を「まあまあ一致している」と読んではいけない。**
#   2人の評定の偏りしだいでは、でたらめに付けても相当な割合が偶然に一致する。
#   偶然一致率は水準数だけでは決まらず、各評定者の評定の分布で決まる。
#   一致率は偶然の一致を含んだ値なので、そのままでは解釈できない。
# ・Cohen's kappaは**偶然の一致を差し引いた**指標である。r1-r2で0.391。
#   慣習的な目安は 0.41-0.60 で「中程度」、0.61-0.80 で「かなり良い」、
#   0.81以上で「ほぼ完全」とされる（Landis & Kochの基準）。
#   0.391は「まずまず」程度で、一致率55%という見かけよりずっと厳しい評価になる。
# ・r1-r3のkappaは0.211とさらに低い。r3は「全体的に甘めに付ける」評定者として
#   作ったので、体系的にずれている。**kappaは体系的なずれに厳しい。**
# ・★ところが**重み付きkappaにすると0.622まで跳ね上がる**。
#   通常のkappaは「1と2」の食い違いも「1と4」の食い違いも同じ1つの不一致として
#   数えるが、順序のある評定ではこれは不当である。隣の段への食い違いは
#   軽い誤りであるべきだからである。重み付きkappaはずれの大きさに応じて
#   罰則を変える。**順序尺度の評定では、重み付きkappaを既定の選択肢にする。**
#   通常のkappaを使うと、一致度を大幅に過小評価する。
# ・ICCは評定を連続量とみなしたときの一致度である。2つの値が出ている点に注意する。
#   ICC2（0.659）は**評定者1人あたり**の信頼性、ICC2k（0.853）は
#   **3人の平均値**の信頼性である。★どちらも「two-way random、絶対一致」
#   という同じモデルの単一評定者版・k人平均版なので、この2つを比べて初めて
#   「平均化することで信頼性がどれだけ上がるか」を正しく測ったことになる
#   （型の違うICC1kと比べると、平均化の効果とモデルの違いが混ざってしまう）。
#   ★どちらを報告するかは、**実際の分析で何を使うか**で決まる。
#   3人の平均値を分析に使うならICC2kを、1人ずつの評定を使うならICC2を報告する。
#   平均値を使うのに単一評定者のICCを報告すると、不当に低く見せることになる。
# ・**評定者を増やすと平均値の信頼性は上がる**（0.659 → 0.853、Spearman-Brownの
#   予言公式が示すとおり）。★ただし「評定者は3人以上必要」とこの結果だけから
#   一般化してはいけない。必要な人数は単一評定者の信頼性・目標とする合成評定の
#   信頼性・測定対象によって変わる。何人必要かはSpearman-Brownの公式で逆算する。
#   信頼区間も併記されており、ICC2は0.457から0.791と幅が広い。60項目でも
#   この程度の精度しか出ない。


■ 概念をつかむ：ICCは分散の割合：

評定値のばらつき（全体の分散）のうち、対象（項目）の違いが占める割合がICCである。項目と評定者のランダム切片をもつ線形混合モデルから分散成分を取り出し、項目の分散÷合計を手計算して、psych::ICC()のICC2と一致することを確かめる。

In [ ]:
# 縦長形式にして、項目と評定者の2つのランダム切片をもつモデルを当てはめる
long <- data.frame(
  item   = factor(rep(seq_len(nrow(ratings)), times = 3)),
  rater  = factor(rep(c("r1", "r2", "r3"), each = nrow(ratings))),
  rating = c(ratings$r1, ratings$r2, ratings$r3)
)
m <- lmer(rating ~ 1 + (1 | item) + (1 | rater), data = long)
vc <- as.data.frame(VarCorr(m))[, c("grp", "vcov")]
print(vc)

v_item <- vc$vcov[vc$grp == "item"]
v_rater <- vc$vcov[vc$grp == "rater"]
v_res <- vc$vcov[vc$grp == "Residual"]
cat("項目の分散の割合 (lmer)    :", round(v_item / (v_item + v_rater + v_res), 3), "\n")
cat("psych::ICC の ICC2         :", round(ICC(ratings)$results[2, "ICC"], 3), "\n")
cat("psych::ICC の ICC3         :", round(ICC(ratings)$results[3, "ICC"], 3), "\n")
cat("評定者の分散を除いた割合   :", round(v_item / (v_item + v_res), 3), "\n")

# --- 実行結果 ---
#        grp       vcov
# 1     item 0.67711750
# 2    rater 0.09660869
# 3 Residual 0.25339007
# 項目の分散の割合 (lmer)    : 0.659
# psych::ICC の ICC2         : 0.659
# psych::ICC の ICC3         : 0.728
# 評定者の分散を除いた割合   : 0.728
#
# 【出力の読み方】
# ・分散は項目0.677、評定者0.097、残差0.253で合計1.027。項目の分散を合計で割ると
#   0.659になり、psych::ICC()のICC2（単一評定者、two-way random、絶対一致）と一致する。
#   ICCは「評定値の分散のうち、対象（項目）の違いが占める割合」である。
# ・分母から評定者の分散を除くと0.728で、これはICC3（一貫性）の値と一致する。
#   評定者の甘辛を誤差に数えるか否かで0.659と0.728に分かれる。型の明記が必要な理由である。
# ・★ICCは対象間の違いと測定のゆらぎの比である。均質な対象ばかりのデータでは低く出る。


■ 自動測定と手動測定の一致：Bland-Altman法：

ここまでは人と人の一致を扱ったが、強制アライメントなど機械と人の一致には別の見せ方が向く。相関係数だけでは系統的なずれ（バイアス）を見落とす。

In [ ]:
set.seed(15)
manual <- rnorm(80, 60, 15)               # 手動計測のVOT
auto <- manual + 4 + rnorm(80, 0, 6)      # 自動計測: +4msの系統的バイアス + ランダム誤差

diff <- auto - manual
bias <- mean(diff)
sd_diff <- sd(diff)
cat("平均バイアス（自動-手動）:", round(bias, 2), "ms\n")
cat("差のSD:", round(sd_diff, 2), "ms\n")
cat("一致の限界: [", round(bias - 1.96*sd_diff, 2), ",", round(bias + 1.96*sd_diff, 2), "]\n")
cat("相関係数（参考、一致度の指標としては不適切）:", round(cor(manual, auto), 3), "\n")

# Bland-Altmanプロット: 差（縦軸）を平均（横軸）に対してプロットする
ggplot(data.frame(avg = (auto+manual)/2, diff = diff), aes(avg, diff)) +
  geom_point(alpha = 0.5) +
  geom_hline(yintercept = bias, color = "blue") +
  geom_hline(yintercept = c(bias - 1.96*sd_diff, bias + 1.96*sd_diff),
             linetype = "dashed", color = "red") +
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果 ---
# 平均バイアス（自動-手動）: 3.04 ms
# 差のSD: 6.74 ms
# 一致の限界: [ -10.17 , 16.25 ]
# 相関係数（参考、一致度の指標としては不適切）: 0.909
#
# 【出力の読み方】
# ・平均バイアス3.04msは「自動測定は手動測定より平均して3.04ms長く出る」という
#   系統的なずれを表す。一致の限界[-10.17, 16.25]は「最大でこれだけ食い違う」という
#   上限ではない。差が正規分布に従うという仮定のもとで、将来の1回の測定の差の
#   およそ95%が収まると期待される区間であり、残りおよそ5%はこの範囲の外に
#   出ることが定義上想定されている。
# ・バイアスとLoAはそれ自体が標本から推定された値であり、固有の不確実性を持つ
#   （バイアスの標準誤差は差のSD/√n、LoAの標準誤差はおよそ差のSD×√(3/n)）。
#   標本数が少ないときは信頼区間を添えて報告する。また同じ話者から複数の差を
#   測っている場合（反復測定）は観測が独立でないため、単純なsd(diff)は
#   ばらつきを過小評価しうる点にも注意する。
# ・相関係数0.909だけを見ていると、このバイアスとばらつきの両方を見落とす。
#   **相関は「一方を定数倍・定数だけずらしても1のまま変わらない」性質を持つため、
#   系統的なバイアスを検出する力がない。** 一致度を主張したいなら相関ではなく
#   Bland-Altman法（バイアスとLoA）かICC（絶対一致の定式化）を使う。
# ・強制アライメントの精度検証も同じ発想でできる。境界のずれ（自動-人手）の
#   分布から、平均（系統的な早まり・遅れ）とSD（ばらつき）を報告する。


■ 2. ★情報を持つ欠測：測定の失敗が条件と相関するとき：

F0の自動抽出は、声門振動が不規則な区間で失敗する。つまり**きしみ声（creaky voice）では失敗しやすい**。そして、きしみ声はF0が低いときに起こりやすい。したがって**「測れなかったデータ」は低いF0に偏っている**。この状況で、測れたデータだけを平均すると何が起きるかを確かめる。

In [ ]:
set.seed(16)
n_speaker <- 20
n_item    <- 40

f0_data <- expand_grid(speaker = sprintf("S%02d", 1:n_speaker), item = 1:n_item) %>%
  mutate(
    cond = rep(c("modal", "creaky"), length.out = n()),
    # 真のF0: creaky条件では25Hz低い
    f0_true = 180 + if_else(cond == "creaky", -25, 0) +
              rnorm(n_speaker, 0, 15)[as.integer(factor(speaker))] +
              rnorm(n(), 0, 12),
    # ★抽出の失敗確率は F0 が低いほど高い（条件ではなくF0の値で決まる）
    p_fail = plogis(-1.0 - 0.10 * (f0_true - 180)),
    failed = rbinom(n(), 1, p_fail),
    f0_obs = if_else(failed == 1, NA_real_, f0_true)
  )

cat("抽出失敗率: modal", round(mean(f0_data$failed[f0_data$cond == "modal"]), 3),
    " / creaky", round(mean(f0_data$failed[f0_data$cond == "creaky"]), 3), "\n")
cat("全体の欠測率:", round(mean(f0_data$failed), 3), "\n\n")

true_m <- mean(f0_data$f0_true[f0_data$cond == "modal"])
true_c <- mean(f0_data$f0_true[f0_data$cond == "creaky"])
obs <- f0_data %>% filter(!is.na(f0_obs))
obs_m <- mean(obs$f0_obs[obs$cond == "modal"])
obs_c <- mean(obs$f0_obs[obs$cond == "creaky"])

cat(sprintf("真の値  : modal %.2f  creaky %.2f  差 %.2f\n", true_m, true_c, true_m - true_c))
cat(sprintf("観測値  : modal %.2f  creaky %.2f  差 %.2f\n", obs_m, obs_c, obs_m - obs_c))
cat(sprintf("差の過小評価: %.1f%%\n", 100 * (1 - (obs_m - obs_c) / (true_m - true_c))))

cat("\n真値で当てはめたモデルの係数  :",
    round(fixef(lmer(f0_true ~ cond + (1 | speaker), data = f0_data))["condmodal"], 2), "\n")
cat("観測値で当てはめたモデルの係数:",
    round(fixef(lmer(f0_obs  ~ cond + (1 | speaker), data = f0_data))["condmodal"], 2), "\n")

# --- 実行結果 ---
# 抽出失敗率: modal 0.28  / creaky 0.652
# 全体の欠測率: 0.466
#
# 真の値  : modal 183.82  creaky 159.93  差 23.89
# 観測値  : modal 190.55  creaky 174.23  差 16.32
# 差の過小評価: 31.7%
#
# 真値で当てはめたモデルの係数  : 23.89
# 観測値で当てはめたモデルの係数: 20.15
#
# 【出力の読み方】
# ・**両群とも観測された平均が真の平均より高い。** modalは183.82→190.55、
#   creakyは159.93→174.23。低いF0のトークンが優先的に失われるので、
#   残ったものだけを平均すれば当然そうなる。
# ・重要なのは**上がり方が群によって違う**ことである。creaky条件のほうが
#   失敗率が高い（0.652 対 0.280）ため、平均の押し上げも大きい
#   （+14.30 対 +6.73）。結果として**条件間の差が23.89から16.32へ、
#   31.7%も縮んでいる**。
# ・これは分析手法の問題ではない。混合効果モデルを使っても係数は
#   23.89から20.15へ縮む。**どんな高度なモデルも、失われたデータは復元できない。**
# ・★この状況を「情報を持つ欠測」（informative missingness）と呼ぶ。
#   欠測が完全にランダム（MCAR）なら、標本が減るだけで推定値は歪まない。
#   しかしここでは**欠測するかどうかが、測ろうとしている値そのものに依存する**。
#   これは統計学でMNAR（missing not at random）と呼ばれる最も厄介な型である。
# ・**音声研究ではこの型の欠測が例外ではなく標準である。** 代表例を挙げる。
#   - F0抽出はきしみ声・ささやき声で失敗する（F0が低い/不規則なときに失敗）
#   - フォルマント推定は高いF0の話者（女性・子ども）で不安定になる
#   - 強制アライメントは発話速度が速い区間や不明瞭な区間でずれる
#   - VOTの測定は、閉鎖が弱い場合に境界を決められない
#   いずれも**測りにくい条件そのものが研究対象**であることが多い。
# ・★対処は3段階で考える。
#   (1) **必ず欠測率を報告する。** 条件ごと・話者ごとに集計して示す。
#       上の例なら「creaky条件で65%が抽出に失敗した」と書く。この一文があれば、
#       読者は結果を割り引いて読める。**黙って除外するのが最悪である。**
#   (2) **欠測が条件と関係しているかを検定する。**
#       glm(failed ~ cond, family = binomial) を当てればよい。
#       関係していれば、単純な除外は使えないと分かる。
#       ★ただしこの検定はMARとMNARを区別しない。「failedが観測共変量condと
#       関連する」ことを示すのみで、これはMAR（観測変数を条件づければ欠測は
#       ランダム）とも矛盾しない所見である。MNAR（欠測が測れなかった値
#       そのものに依存する）は定義上、観測データから直接検証できない仮定
#       であり、感度分析でしか扱えない。
#   (3) **失敗そのものを結果として扱うことを検討する。**
#       「creaky条件では65%でF0が抽出できなかった」という事実は、
#       きしみ声の存在を示す立派な証拠である。測れた分だけを平均するより、
#       抽出可否を二値の応答としてGLMM（第6章）にかけるほうが、
#       研究の問いに答えていることがある。
# ・★最後の点が最も重要である。**測定の失敗をノイズとして捨てるのではなく、
#   データとして読む。** 測れなかったことにも意味がある。


■ 演習：

1. 欠測が条件と関係しているかを `glm(failed ~ cond, family = binomial)` で検定せよ。
2. 抽出失敗を応答変数とするGLMMを当てはめ、「creaky条件では抽出に失敗しやすい」ことを結果として報告する形にせよ。
3. 評定者 r3 を除いて ICC を計算し直し、値がどう変わるかを確かめよ。

■ 演習の解答：

In [ ]:
# 1. 欠測が条件に依存しているかを検定する。
glm_miss <- glm(failed ~ cond, family = binomial, data = f0_data)
print(round(summary(glm_miss)$coefficients, 4))
cat("オッズ比:", round(exp(coef(glm_miss)["condmodal"]), 3),
    "→ modal条件はcreaky条件に比べて失敗のオッズがこの倍率\n")

# 2. 「測れなかったこと」自体を結果として分析する。
glmm_miss <- glmer(failed ~ cond + (1 | speaker), family = binomial, data = f0_data)
cat("\n抽出失敗のGLMM:\n")
print(round(summary(glmm_miss)$coefficients, 4))
cat("creaky条件の失敗確率:", round(plogis(fixef(glmm_miss)["(Intercept)"]), 3), "\n")
cat("modal条件の失敗確率  :", round(plogis(sum(fixef(glmm_miss))), 3), "\n")

# 3. 甘めの評定者r3を除いてICCを計算する。
cat("\n3人全員のICC(単一評定者):",
    round(ICC(ratings)$results[2, "ICC"], 3), "\n")
cat("r1,r2のみのICC(単一評定者):",
    round(ICC(ratings[, c("r1","r2")])$results[2, "ICC"], 3), "\n")

# --- 実行結果 ---
#             Estimate Std. Error  z value Pr(>|z|)
# (Intercept)   0.6300     0.1050   6.0003        0
# condmodal    -1.5745     0.1531 -10.2871        0
# オッズ比: 0.207 → modal条件はcreaky条件に比べて失敗のオッズがこの倍率
#
# 抽出失敗のGLMM:
#             Estimate Std. Error  z value Pr(>|z|)
# (Intercept)   0.8533     0.3053   2.7952   0.0052
# condmodal    -2.0882     0.1918 -10.8863   0.0000
# creaky条件の失敗確率: 0.701
# modal条件の失敗確率  : 0.225
#
# 3人全員のICC(単一評定者): 0.659
# r1,r2のみのICC(単一評定者): 0.769
#
# 【出力の読み方】
# ・1問目: condmodal の係数は −1.5745、p はほぼ0。オッズ比0.207は
#   「modal条件では失敗のオッズがcreaky条件の約5分の1」という意味である。
#   **欠測は明確に条件と結びついている。** したがって
#   「測れた分だけを分析する」という方針はこの時点で使えないと分かる。
#   ★この検定は1行で書けるのに、実際にはほとんど行われない。
#   欠測を見つけたら、まずこれを走らせる習慣をつけたい。
#   ★ただしこの結果だけでは「MNARである」とは言えない。示せたのは欠測が
#   観測変数condと関連していること（MARと矛盾しない所見）までであり、
#   欠測が測れなかった値自体に依存するかどうかは観測データから検証できない。
# ・2問目: 失敗そのものを応答としたGLMMでは、creaky条件の失敗確率が0.701、
#   modal条件が0.225と推定された。**この2つの数値は、それ自体が
#   報告に値する結果である。** 「creaky条件では約70%でF0が抽出できなかった」
#   という記述は、きしみ声が実際に生じていたことの客観的な証拠になる。
# ・話者のランダム切片を入れているので、「特定の話者だけが失敗を稼いでいる」
#   可能性も調整済みである。切片のp = 0.0052 は
#   「creaky条件の失敗確率が50%と異なる」ことを示す。
# ・★研究の書き方が変わる点に注目したい。単純な報告なら
#   「F0の差は16.3Hzだった（真値は23.9Hz）」と、31.7%過小評価した値を
#   そのまま書いてしまう。しかし欠測を分析すれば、
#   「creaky条件では70%で抽出に失敗しており、測れたトークンは
#   相対的にF0が高いものに偏る。観測された16.3Hzという差は、
#   この偏りの方向から見て過小評価されている可能性が高い」と書ける。
#   **同じデータから、はるかに誠実で情報量の多い記述が得られる。**
#   ★ただし「観測値は真の差の下限である」と断定してはいけない。
#   今回のシミュレーションは欠測メカニズムを一方向に設定したため
#   たまたまこの向きにバイアスがかかったが、実データでは欠測メカニズムの
#   向きは観測データだけからは分からない（MNARの識別不能性）ため、
#   一般には「下限」という数量的な保証はできない。書けるのは
#   バイアスの方向についての仮説までであり、裏付けには感度分析が要る。
# ・3問目: 甘めの評定者r3を外すと、単一評定者のICCは0.659から0.769へ上がる。
#   r3が体系的にずれていたぶん、一致度が下がっていたことが確認できる。
# ・★ただし**「一致度が上がるから外す」という判断はしてはいけない。**
#   それはデータを結果に合わせて選ぶことであり、第12章で見た
#   「有意になった比較だけを報告する」のと同じ問題である。
#   評定者を外してよいのは、手順の誤解や作業ミスなど、
#   **一致度とは独立した理由が確認できたとき**に限る。
#   外した場合は、外した理由と、外す前後の両方の値を報告する。
# ・（`r1,r2のみ` のICCを計算する際に "boundary (singular) fit" という警告が
#   出ることがある。評定者2人だと分散成分の推定が境界に貼り付きやすいためで、
#   ICCの値そのものは使える。第5章で扱ったsingular fitと同じ現象である。）
